
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Multi Flow SDP with Liquid Clustering and Data Quality

## Overview

This demonstration showcases how to build a robust incremental data pipeline using Apache Spark™ Declarative Pipelines (SDP) to consolidate data from multiple subsidiaries (data sources) into a single target streaming table. 

You'll work with three fictional company subsidiaries: Bright Home, Lumina Sports, and Northstar Outfitters, each producing transaction data in different formats (`CSV` and `JSON`). The demo illustrates how to overcome common pipeline challenges including multiple flows into a single table, schema mismatches, data quality issues, and performance optimization requirements.

Through hands-on implementation, you'll create a complete medallion architecture pipeline that incrementally ingests multiple data sources into a single bronze table using flows, applies data quality constraints and transformations in the silver layer with liquid clustering optimization, and creates business intelligence materialized views in the gold layer. 

## Learning Objectives

By the end of this demonstration, you will be able to:
- **Ingest multiple data sources into one bronze table** using Spark Declarative Pipelines having different file formats like CSV and JSON.

- **Standardize schemas in the bronze layer** and map correct data types in silver layer to resolve differences across source systems.

- **Add data quality checks and enable liquid clustering** in the silver tables to enforce basic rules and improve query performance.

- **Build incremental materialized views** in the gold layer that refresh automatically and provide ready-to-use analytics.

- **Run and monitor the full pipeline run** across bronze, silver, and gold, including incremental loads and data lineage tracking.

### Multi Flow Pipeline Demonstration Overview
In this demonstration, you'll build a Apache Spark™ Declarative Pipeline that performs the full medallion flow from raw ingestion of multiple raw data sources to curated analytics

1. **Use multiple flows (3)** to incrementally ingest files from three cloud storage locations and write into a single bronze table. 
    - Each volume is a daily orders drop for a specific subsidiary that the company owns. We want all this data ingested into a single table for overall analysis.
2. **Define and build the silver table** with a clean schema, apply basic data quality constraints, and **enable liquid clustering** for query performance as the data continues to grow.
3. **Create gold materialized views** that automatically refresh and provide ready-to-use analytics.

![Multi Flow Pipeline Overview](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/multi_flow_demo_pipeline_overview.png)

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 4**  
  - [How to select an environment version](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V4**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size:1.1em;">
    Option 1 - Databricks Academy Provided Workspace (Vocareum Workspace)
  </strong>
  <details>
  <div style="color:#333;">

If you are running this notebook in a <strong>Databricks Academy provided Vocareum workspace</strong>, your Unity Catalog catalog is already created for you.

Your catalog name matches your Vocareum username and looks like: <strong>labuser12345</strong> (series of unique numbers)
  </div>
  </details>
</div>


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size:1.1em;">
    Option 2 - Other Workspaces or Databricks Free Edition
  </strong>
  <details>
  <div style="color:#333;">

If you are running this notebook in your own Databricks workspace or Databricks Free Edition, the setup will
<strong>create a Unity Catalog catalog and schema for you</strong>. **Create catalog permission is required.**

The catalog name is derived from your Databricks username and follows this pattern: <strong>labuser_username</strong>
  </div>
  </details>
</div>

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Do Not Run in Production Environments</strong>
  <div style="color:#333;">
  <ul>
      <li>Only run this notebook in <strong>development or sandbox workspaces</strong>.</li>
      <li>Do not run this in production environments. The setup script creates a catalog and schemas in your workspace.</li>
  </ul>
  </div>
</div>

## A. Setup

### A1. Access Marketplace Data
<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">

  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Option 1 - Databricks Academy Provided Workspace (Vocareum Workspace)
  </strong>
<details>
  <div style="color:#333;">
  If you are running this lab in a <strong>Databricks Academy provided Vocareum workspace</strong>, the share is already installed and available as <strong>dbacademy_retail</strong>. Please use this as the value for the <code>your_marketplace_share_catalog_name</code> variable below.
  </div>
</details>
</div>


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Option 2 - Other Workspaces or Databricks Free Edition
  </strong>
<details>
  <div style="color:#333;">

  If you are running this in your own Workspace, complete the following steps to get your own copy of the Marketplace data. If you already have this share simply add that name to the variable below.

1. Open **Databricks Marketplace** in a new tab.  

2. Search for `Simulated Retail Customer Data`.  

3. Select the tile titled **Simulated Retail Customer Data (Databricks provided)**.  

4. Click **Get instant access**.  

5. **Enter a unique catalog name** for your share to avoid receiving a duplicate catalog error in shared Workspaces. For example: `dbacademy_retail_yourname`.  

6. Review and accept the terms, then click **Get instant access** to complete the setup.

7. Update the variable `your_marketplace_share_catalog_name` in cell below to point to your shared catalog from Marketplace.
  </div>
</details>
</div>

In [0]:
## Update the variable below to reference your marketplace catalog name

## NOTE: If you are using Vocareum, use the value 'dbacademy_retail' catalog below
your_marketplace_share_catalog_name = 'dbacademy_retail'

### A2. Configure Your Catalog and Schema

1. Run the cell below to initialize your environment. 

    This setup step does the following:

    - **Assumes you have permission to create a catalog** when running outside of a Databricks provided Vocareum workspace
    - Create three schemas in your specified catalog:  
        - **multi_flow_1_bronze**
        - **multi_flow_2_silver**
        - **multi_flow_3_gold**  
    - Creates a three volumes in your **YOUR_LABUSER_CATALOG.multi_flow_1_bronze** schema and adds a single JSON file in each volume.
    - Checks your specified Serverless compute version

This ensures that all schemas, tables and objects are created in your catalog.

> **Important:** You must have permission to create catalogs in your own non Vocareum workspace. If you do not have the required permissions, this step will fail. Review the note below before continuing.


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">

  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Troubleshooting Setup - Missing Create Catalog Permissions
  </strong>
<details>
  <div style="color:#333;">

If you do not have permission to create a new catalog but already have one available, you can explicitly specify an existing catalog by using the `catalog_forced` argument in the `build_user_catalog_name` function.

This function is defined in the notebook: `./Includes/Classroom-Setup-multiple-flows`

  </div>
</details>
</div>





In [0]:
%run ./Includes/Classroom-Setup-multiple-flows

Learner is not using a Databricks Academy provided Vocareum Workspace.
Using learner set catalog: dbx_gen_ai
✅ Catalog 'dbx_gen_ai' already exists in your Workspace. Using this catalog.


✅ All required variables are defined and have the following values:
  - your_marketplace_share_catalog_name = dbacademy_retail




True

Creating schema dbx_gen_ai.multi_flow_1_bronze...
Creating schema dbx_gen_ai.multi_flow_2_silver...
Creating schema dbx_gen_ai.multi_flow_3_gold...
Creating volume bright_home_orders...
Creating volume lumina_sports_orders...
Creating volume northstar_outfitters_orders...
Creating Python and SQL variables to volume paths: bright_home_orders_path, lumina_sports_orders_path, northstar_outfitters_orders_path
Reset volumes by deleting files

Searching for files in /Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders/ volume to delete prior to creating files...
Deleting file: /Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders/bsh_orders_2025-11-01.csv

Searching for files in /Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders/ volume to delete prior to creating files...
Deleting file: /Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders/lms_orders_2025-11-01.csv

Searching for files in /Volumes/dbx_gen_ai/multi_flow_1_bronze/northstar_outfitters_orders/ volume

Your Marketplace Share Catalog:,
Your Catalog:,
Your Schemas:,
Your Data Source Volume Path:,
Your Data Source Volume Path:,
Your Data Source Volume Path:,


2. Run the cell below to view the value of the `my_vol_path` variable.

   Confirm that the value references your **your-catalog.multi_flow_1_bronze** path. This will be used to dynamically reference your source volumes throughout this demonstration.


In [0]:
print(my_vol_path)

/Volumes/dbx_gen_ai/multi_flow_1_bronze


## B. Explore the Source Volumes for Ingesting Multiple Flows into a Single Target

Before building multiple flows that write to a single target streaming table, start by exploring the raw source data stored in in the three volumes.  

Each **volume** represents a **separate sales system** for a different subsidiary within our fictional company:

- **B1.** Bright Home Orders volume (`CSV` files)  
- **B2.** Lumina Sports Orders volume (`CSV` files)  
- **B3.** Northstar Outfitters Orders volume (`JSON` files)

### B1. Bright Home Orders Volume

1. View the files in the **multi_flow_1_bronze.bright_home_orders** volume.

   Notice that only one `CSV` file currently exists in this volume for the sales on **2025-11-01**.


In [0]:
spark.sql(f"LIST '{my_vol_path}/bright_home_orders'").display()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8786736017830667>, line 1
----> 1 spark.sql(f"LIST '{my_vol_path}/bright_home_orders'").display()

NameError: name 'my_vol_path' is not defined

2. Explore the raw data for **bright_home_orders**. The cell below performs the following:

   a. Counts the number of records in the file within the volume.  
   b. Describes the default ingestion data types for each column of the `CSV` file.  
   c. Previews the data.  

In the output, notice the following:
- **157** rows are present in this file.  
- The schema is inferred and returns a variety of data types.  
- This is simple sales order data from the company's **Bright Home** subsidiary.


In [0]:
# a. Row count
df_count = spark.sql(f"""
    SELECT count(*) AS TotalRows
    FROM read_files('{my_vol_path}/bright_home_orders')
""")
display(df_count)

# b. Schema
df_schema = spark.sql(f"""
    DESCRIBE SELECT * 
    FROM read_files('{my_vol_path}/bright_home_orders')
""")
display(df_schema)

# c. Preview rows
df_preview = spark.sql(f"""
    SELECT *
    FROM read_files('{my_vol_path}/bright_home_orders')
    LIMIT 5
""")
display(df_preview)

TotalRows
157


col_name,data_type,comment,metadata
subsidiary_id,string,null,null
order_id,string,null,null
order_timestamp,timestamp,null,null
customer_id,string,null,null
region,string,null,null
country,string,null,null
city,string,null,null
channel,string,null,null
sku,string,null,null
category,string,null,null


subsidiary_id,order_id,order_timestamp,customer_id,region,country,city,channel,sku,category,qty,unit_price,discount_pct,coupon_code,total_amount,order_date,_rescued_data
BSH,BSH_ORD_20251101_0001,2025-11-01T19:33:23.000Z,CUST_02370,EU,UK,London,online,SKU-3001,fitness,2,9.99,10,null,17.98,2025-11-01,null
BSH,BSH_ORD_20251101_0002,2025-11-01T04:40:39.000Z,CUST_01411,NA,US,Seattle,online,SKU-2001,home,3,49.99,15,null,127.47,2025-11-01,null
BSH,BSH_ORD_20251101_0003,2025-11-01T12:15:53.000Z,CUST_00486,NA,US,Austin,online,SKU-5001,outdoor,1,29.99,0,null,29.99,2025-11-01,null
BSH,BSH_ORD_20251101_0004,2025-11-01T02:27:41.000Z,CUST_04829,NA,US,Austin,marketplace,SKU-3001,fitness,4,29.99,5,null,113.96,2025-11-01,null
BSH,BSH_ORD_20251101_0005,2025-11-01T02:11:24.000Z,CUST_01825,EU,DE,Berlin,marketplace,SKU-2001,home,4,99.99,5,null,379.96,2025-11-01,null


### B2. Lumina Sports Orders Volume

1. View the files in the **multi_flow_1_bronze.lumina_sports_orders** volume.

   Notice that only one `CSV` file currently exists in this volume for the sales on **2025-11-01**.

In [0]:
spark.sql(f"LIST '{my_vol_path}/lumina_sports_orders'").display()

path,name,size,modification_time
/Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders/lms_orders_2025-11-01.csv,lms_orders_2025-11-01.csv,14248,1791030180000


2. Explore the raw data for **Lumina Sports**. The cell below performs the following:

   a. Counts the number of records in the file within the volume.  
   b. Describes the default ingestion data types for each column of the `CSV` file.  
   c. Previews the data.  

In the output, notice the following:
- **110** rows are present in this file.  
- The schema is inferred and returns a variety of data types.  
- This is simple sales order data from the company's **Lumina Sports** subsidiary.


In [0]:
# a. Row count
df_count = spark.sql(f"""
    SELECT count(*) AS TotalRows
    FROM read_files('{my_vol_path}/lumina_sports_orders')
""")
display(df_count)

# b. Schema
df_schema = spark.sql(f"""
    DESCRIBE SELECT *
    FROM read_files('{my_vol_path}/lumina_sports_orders')
""")
display(df_schema)

# c. Full preview
df_preview = spark.sql(f"""
    SELECT *
    FROM read_files('{my_vol_path}/lumina_sports_orders')
    LIMIT 5
""")
display(df_preview)

TotalRows
110


col_name,data_type,comment,metadata
subsidiary_id,string,null,null
order_id,string,null,null
order_timestamp,timestamp,null,null
customer_id,string,null,null
region,string,null,null
country,string,null,null
city,string,null,null
channel,string,null,null
sku,string,null,null
category,string,null,null


subsidiary_id,order_id,order_timestamp,customer_id,region,country,city,channel,sku,category,qty,unit_price,discount_pct,coupon_code,total_amount,order_date,_rescued_data
LMS,LMS_ORD_20251101_0001,2025-11-01T10:31:18.000Z,CUST_04487,NA,US,Austin,online,SKU-4001,electronics,3,14.99,15,null,38.22,2025-11-01,null
LMS,LMS_ORD_20251101_0002,2025-11-01T05:53:03.000Z,CUST_02804,EU,UK,London,marketplace,SKU-5001,outdoor,4,9.99,15,null,33.97,2025-11-01,null
LMS,LMS_ORD_20251101_0003,2025-11-01T21:41:50.000Z,CUST_04968,EU,DE,Berlin,marketplace,SKU-1001,apparel,1,99.99,10,null,89.99,2025-11-01,null
LMS,LMS_ORD_20251101_0004,2025-11-01T09:22:26.000Z,CUST_00537,EU,DE,Berlin,mobile_app,SKU-3001,fitness,4,49.99,15,null,169.97,2025-11-01,null
LMS,LMS_ORD_20251101_0005,2025-11-01T09:22:23.000Z,CUST_03596,EU,DE,Berlin,marketplace,SKU-5001,outdoor,1,29.99,0,null,29.99,2025-11-01,null


### B3. Northstar Outfitters Orders Volume

1. View the files in the **multi_flow_1_bronze.northstar_outfitters_orders** volume.

   Notice that only one `JSON` file currently exists in this volume for the sales on **2025-11-01**.

In [0]:
spark.sql(f"LIST '{my_vol_path}/northstar_outfitters_orders'").display()

path,name,size,modification_time
/Volumes/dbx_gen_ai/multi_flow_1_bronze/northstar_outfitters_orders/nso_orders_2025-11-01.json,nso_orders_2025-11-01.json,68071,1791030185000


2. Explore the raw data for **Northstar Outfitters**. The cell below performs the following:

   a. Counts the number of records in the file within the volume.  
   b. Describes the default ingestion data types for each column of the `JSON` file.  
   c. Previews the data.  

In the output, notice the following:
- **182** rows are present in this file.  
- The schema is inferred and returns a variety of data types.  
- This is simple sales order data from the company's **Northstar Outfitters** subsidiary.


In [0]:
# a. Row count
df_count = spark.sql(f"""
    SELECT count(*) AS TotalRows
    FROM read_files('{my_vol_path}/northstar_outfitters_orders')
""")
display(df_count)

# b. Schema
df_schema = spark.sql(f"""
    DESCRIBE SELECT *
    FROM read_files('{my_vol_path}/northstar_outfitters_orders')
""")
display(df_schema)

# c. Full preview
df_preview = spark.sql(f"""
    SELECT *
    FROM read_files('{my_vol_path}/northstar_outfitters_orders')
    LIMIT 5
""")
display(df_preview)

TotalRows
182


col_name,data_type,comment,metadata
category,string,null,null
channel,string,null,null
city,string,null,null
country,string,null,null
coupon_code,string,null,null
customer_id,string,null,null
discount_pct,bigint,null,null
order_date,string,null,null
order_id,string,null,null
order_timestamp,string,null,null


category,channel,city,country,coupon_code,customer_id,discount_pct,order_date,order_id,order_timestamp,qty,region,sku,subsidiary_id,total_amount,unit_price,_rescued_data
home,online,Austin,US,null,CUST_04697,5,2025-11-01,NSO_ORD_20251101_0001,2025-11-01T18:57:40Z,2,NA,SKU-2001,NSO,18.98,9.99,null
outdoor,online,Berlin,DE,null,CUST_04660,15,2025-11-01,NSO_ORD_20251101_0002,2025-11-01T14:22:08Z,3,EU,SKU-5001,NSO,254.97,99.99,null
outdoor,mobile_app,Denver,US,null,CUST_03530,15,2025-11-01,NSO_ORD_20251101_0003,2025-11-01T12:54:07Z,3,NA,SKU-5001,NSO,76.47,29.99,null
apparel,online,London,UK,null,CUST_04619,0,2025-11-01,NSO_ORD_20251101_0004,2025-11-01T03:16:15Z,3,EU,SKU-1001,NSO,299.97,99.99,null
electronics,mobile_app,Austin,US,null,CUST_04903,10,2025-11-01,NSO_ORD_20251101_0005,2025-11-01T23:58:20Z,2,NA,SKU-4001,NSO,179.98,99.99,null


### B4. Raw Data Exploration Summary

Below is a quick overview of what we discovered after examining the raw source volumes. 

#### Raw Cloud Storage Overview
Each source volume contains one file with a small number of sales for our demonstration.

| Source Data (volume)            | File Format | # of Rows | # of Files | Sales Date |
|---------------------------------|-------------|-----------|------------|------------|
| Bright Home Orders              | CSV         | 157       | 1          | 2025-11-01|
| Lumina Sports Orders            | CSV         | 110       | 1          | 2025-11-01|
| Northstar Outfitters Orders     | JSON        | 182       | 1          | 2025-11-01|


<br>

#### Schema Differences Comparison and Issues

- Each raw data source uses its own structure (`CSV` or `JSON`).  
- When ingesting `CSV`, the inferred column types often differ from `JSON`.  
- Since each format infers schemas independently, combining these three flows into a single target table will introduce **schema mismatches that lead to ingestion conflicts**.

| **Column Name**     | **bright_home_orders (CSV)** | **lumina_sports_orders (CSV)** | **northstar_outfitters_orders (JSON)** |
|---------------------|------------------------|---------------------------|---------------------------------|
| subsidiary_id       | string                 | string                    | string                          |
| order_id            | string                 | string                    | string                          |
| **order_timestamp** | **timestamp**          | **timestamp**             | **string**                      |
| customer_id         | string                 | string                    | string                          |
| region              | string                 | string                    | string                          |
| country             | string                 | string                    | string                          |
| city                | string                 | string                    | string                          |
| channel             | string                 | string                    | string                          |
| sku                 | string                 | string                    | string                          |
| category            | string                 | string                    | string                          |
| **qty**             | **int**                | **int**                   | **bigint**                      |
| unit_price          | double                 | double                    | double                          |
| **discount_pct**    | **int**                | **int**                   | **bigint**                      |
| coupon_code         | string                 | string                    | string                          |
| total_amount        | double                 | double                    | double                          |
| **order_date**      | **date**               | **date**                  | **string**                      |
| _rescued_data       | string                 | string                    | string                          |

<br>

#### Goal: Ingest All Raw Source Files Into One Bronze Streaming Table

To successfully combine all three sources into a single bronze streaming table, we will need to standardize the schema. 

To do this, we will **ingest every column as a `STRING`** into the bronze table.

This avoids data type conflicts between `CSV` and `JSON` files, since each format infers types differently. Normalizing everything to `STRING` keeps the bronze layer predictable, prevents ingestion failures, and lets us apply the correct data types later in the silver layer.

## C. Create the Spark Declarative Pipeline

### C1. Create a Apache Spark™ Declarative Pipeline using the Lakeflow Pipelines Editor
Complete the following steps to create your Spark Declarative Pipeline:

1. In the main navigation pane, right-click **Jobs & Pipelines** and select **Open link in New Tab**.  

2. In the new tab, select **Create → ETL Pipeline**.  

3. At the top, complete the following:
   - Name your pipeline `demo_multi_flow_yourname`
   - Select your default **catalog** and **schema**:  
        - **Catalog:** The catalog you specified for this notebook  
        - **Schema:** **multi_flow_1_bronze**  
      **NOTE:** Clear the selected schema using the cross icon to view all schemas.

4. Rename the **transformations** folder to `ingest_multiple_flows`.

5. Rename the **my_transformation.py** file to `flow_ingestion.sql`.

6. Leave the **Lakeflow Pipelines Editor** page open.

#### Checkpoint
![Create SDP Checkpoint](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/pipeline_creation.png)

## D. Using Multiple Flows to Write to a Single Target

In many enterprise environments, data arrives from several systems that must be consolidated into a single table for downstream processing.

In this example, the company has **three subsidiaries, each producing transaction data in slightly different raw file formats**. 

With Spark Declarative Pipelines, you can define multiple flows that write to the same target streaming table, allowing all raw files in cloud storage to be **incrementally ingested into one unified destination**.

### D1. Create the Bronze Target Table

1. Start by creating the **bronze streaming table** that will serve as the landing zone for all incoming transaction data. 

    This table ingests every column as `STRING` to ensure compatibility across the different source systems.


    Copy the SQL code below and paste it into your `flow_ingestion.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
------------------------------------------
-- CREATE THE BRONZE TABLE STRUCTURE
------------------------------------------
CREATE OR REPLACE STREAMING TABLE multi_flow_1_bronze.orders_bronze_flows_demo
(
  subsidiary_id   STRING,
  order_id        STRING,
  order_timestamp STRING,
  customer_id     STRING,
  region          STRING,
  country         STRING,
  city            STRING,
  channel         STRING,
  sku             STRING,
  category        STRING,
  qty             STRING,
  unit_price      STRING,
  discount_pct    STRING,
  coupon_code     STRING,
  total_amount    STRING,
  order_date      STRING,
  source_file     STRING,   -- Added by the _metadata column to return the source file name
  file_mod_time   TIMESTAMP -- Added by the _metadata column to return file modification time of the file. Returns a consistent value
)
COMMENT "Creates a single bronze streaming table with orders from all subsidiaries using multiple flows."
TBLPROPERTIES (
  'pipelines.reset.allowed' = false    -- prevent full table refreshes on the bronze table
);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the code**
- The `COMMENT` clause adds descriptive metadata to the table for documentation purposes.

- The `TBLPROPERTIES` statement configures the following setting:
  - **Reset Protection**: The `'pipelines.reset.allowed' = false` property prevents full refreshes on the streaming table, which helps avoid accidentally removing checkpoints and truncating the streaming table data.

#### IMPORTANT: Understanding Full Table Refresh Protection

This protection is particularly important when your raw data source automatically removes files after a certain timeframe. Without this setting, data that is no longer present in the source directory would not be reingested into the target table during a **Run pipeline with full table refresh** operation.

**NOTE:** For guidance on when to use full refreshes, see the [Should I use a full refresh?](https://docs.databricks.com/aws/en/ldp/updates#should-i-use-a-full-refresh) documentation.

### D2. Configure the Pipeline Parameters

1. Run the cell below to retrieve the key value pairs needed to set your pipeline configuration parameters for each **raw data source volume**.

In [0]:
config_parameters = [
    ('bright_home_orders_source',        f'{my_vol_path}/bright_home_orders'),
    ('lumina_sports_orders_source',      f'{my_vol_path}/lumina_sports_orders'),
    ('northstar_outfitters_orders_source', f'{my_vol_path}/northstar_outfitters_orders')
]

for key, value in config_parameters:
    print(f"Key: {key}\nValue: {value}\n")

Key: bright_home_orders_source
Value: /Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders

Key: lumina_sports_orders_source
Value: /Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders

Key: northstar_outfitters_orders_source
Value: /Volumes/dbx_gen_ai/multi_flow_1_bronze/northstar_outfitters_orders



2. Copy the paths above and add each one by one as a configuration parameter in your **Spark Declarative Pipeline**.

   This will allow your pipeline to reference each volume through parameters.

   a. Select **Settings** in your pipeline tab.  

   b. Under **Configuration**, select **Add configuration**. 

   c. For each **Key**, enter the key name shown above.  

   d. For each **Value**, enter the corresponding volume path.  

   e. Select **Save**.

   **NOTE:** For more details on configuration parameters, see the Databricks documentation: [Use parameters with Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/parameters)


#### Checkpoint (your path will vary)
<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/checkpoint_config_params.png" alt="Config Parameter Checkpoint" width="600">


### D3. Configure Flow from the Store Bright Home Orders Volume

1. Copy the code below and paste into your `flow_ingestion.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
------------------------------------------
-- BRONZE FLOW - BRIGHT HOME
------------------------------------------
-- Read CSV files from the bright_home_orders volume
CREATE FLOW bright_home_orders_flow
AS INSERT INTO multi_flow_1_bronze.orders_bronze_flows_demo BY NAME
SELECT
  CAST(subsidiary_id AS STRING) AS subsidiary_id,
  CAST(order_id AS STRING) AS order_id,
  CAST(order_timestamp AS STRING) AS order_timestamp,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(region AS STRING) AS region,
  CAST(country AS STRING) AS country,
  CAST(city AS STRING) AS city,
  CAST(channel AS STRING) AS channel,
  CAST(sku AS STRING) AS sku,
  CAST(category AS STRING) AS category,
  CAST(qty AS STRING) AS qty,
  CAST(unit_price AS STRING) AS unit_price,
  CAST(discount_pct AS STRING) AS discount_pct,
  CAST(coupon_code AS STRING) AS coupon_code,
  CAST(total_amount AS STRING) AS total_amount,
  CAST(order_date AS STRING) AS order_date,
  _metadata.file_name AS source_file,
  _metadata.file_modification_time AS file_mod_time
FROM STREAM read_files(
    '${bright_home_orders_source}',   -- Uses the configuration parameter to point to the bright_home_orders volume
    format => 'csv',
    header => true
);
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>


**Review the code**
- All business columns are cast to `STRING` so this flow aligns with the unified bronze schema, while the **metadata** columns keep their native types.  
- The **metadata columns** capture the source **file name and modification time**, which helps with lineage and debugging.  
- The `FROM STREAM read_files('${bright_home_orders_source}', ...)` clause uses your configuration parameter to reference the volume path and relies on Auto Loader for incremental ingestion.

### D4. Configure Flow from Store Lumina Sports Orders Volume

1. Copy the code below and paste into your `flow_ingestion.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
------------------------------------------
-- BRONZE FLOW - LUMINA SPORTS
------------------------------------------
-- Read CSV files from the lumina_sports_orders volume
CREATE FLOW lumina_sports_orders_flow
AS INSERT INTO multi_flow_1_bronze.orders_bronze_flows_demo BY NAME
SELECT
  CAST(subsidiary_id AS STRING) AS subsidiary_id,
  CAST(order_id AS STRING) AS order_id,
  CAST(order_timestamp AS STRING) AS order_timestamp,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(region AS STRING) AS region,
  CAST(country AS STRING) AS country,
  CAST(city AS STRING) AS city,
  CAST(channel AS STRING) AS channel,
  CAST(sku AS STRING) AS sku,
  CAST(category AS STRING) AS category,
  CAST(qty AS STRING) AS qty,
  CAST(unit_price AS STRING) AS unit_price,
  CAST(discount_pct AS STRING) AS discount_pct,
  CAST(coupon_code AS STRING) AS coupon_code,
  CAST(total_amount AS STRING) AS total_amount,
  CAST(order_date AS STRING) AS order_date,
  _metadata.file_name AS source_file,
  _metadata.file_modification_time AS file_mod_time
FROM STREAM read_files(
  '${lumina_sports_orders_source}',   -- Uses the configuration parameter to point to the lumina sports volume
  format => 'csv',
  header => true
);
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>


**Review the code**
- All business columns are cast to `STRING` so this flow aligns with the unified bronze schema, while the **metadata** columns keep their native types.  
- The **metadata columns** capture the source **file name and modification time**, which helps with lineage and debugging.  
- The `FROM STREAM read_files('${lumina_sports_orders_source}', ...)` clause uses your configuration parameter to reference the volume path and relies on Auto Loader for incremental ingestion.

### D5. Configure Flow from Northstar Outfitters Orders Volume

1. Copy the code below and paste into your `flow_ingestion.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
------------------------------------------
-- BRONZE FLOW - NORTHSTAR OUTFITTERS
------------------------------------------
-- Read JSON files from the northstar_outfitters_orders volume
CREATE FLOW northstar_outfitters_orders_flow
AS INSERT INTO multi_flow_1_bronze.orders_bronze_flows_demo BY NAME
SELECT
  CAST(subsidiary_id AS STRING) AS subsidiary_id,
  CAST(order_id AS STRING) AS order_id,
  CAST(order_timestamp AS STRING) AS order_timestamp,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(region AS STRING) AS region,
  CAST(country AS STRING) AS country,
  CAST(city AS STRING) AS city,
  CAST(channel AS STRING) AS channel,
  CAST(sku AS STRING) AS sku,
  CAST(category AS STRING) AS category,
  CAST(qty AS STRING) AS qty,
  CAST(unit_price AS STRING) AS unit_price,
  CAST(discount_pct AS STRING) AS discount_pct,
  CAST(coupon_code AS STRING) AS coupon_code,
  CAST(total_amount AS STRING) AS total_amount,
  CAST(order_date AS STRING) AS order_date,
  _metadata.file_name AS source_file,
  _metadata.file_modification_time AS file_mod_time
FROM STREAM read_files(
  '${northstar_outfitters_orders_source}',  -- Uses the configuration parameter to point to the northstar volume
  format => 'json'
);
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>


**Review the code**
- All business columns are cast to `STRING` so this flow aligns with the unified bronze schema, while the **metadata** columns keep their native types.  
- The **metadata columns** capture the source **file name and modification time**, which helps with lineage and debugging.  
- The `FROM STREAM read_files('${northstar_outfitters_orders_source}', ...)` clause uses your configuration parameter to reference the volume path and relies on Auto Loader for incremental ingestion.

### D6. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm it runs successfully. 

2. Explore the run in the Lakeflow Pipelines Editor.
  - Confirm **449** rows were ingested into the bronze table from the three volumes (**157 + 110 + 182**)
  - Preview the data in the editor (Select **orders_bronze_flows_demo** -> **Data** tab). Notice data from each volume was incrementally ingested into the bronze table.

> **TROUBLESHOOTING:** If your pipeline does not run successfully, confirm that your volumes were created and that your configuration parameters are set correctly.

#### Checkpoint

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/checkpoint_bronze_flows.png" alt="Bronze Flow" width="1200">

### D7. Query the Bronze Streaming Table
1. The query below groups records by the captured **source_file** column so you can see the number of ingested rows per file.

    Review how many files were ingested from each source confirming each cloud storage flow was ingested successfully.

#### Checkpoint
| Source File                 | Total Ingested by Source  |
|-----------------------------|---------------------------|
| nso_orders_2025-11-01.json  | 182                       |
| bsh_orders_2025-11-01.csv   | 157                       |
| lms_orders_2025-11-01.csv   | 110                       |

In [0]:
%sql
SELECT source_file, count(*) AS `Total Ingested by Source`
FROM multi_flow_1_bronze.orders_bronze_flows_demo
GROUP BY source_file

source_file,Total Ingested by Source
nso_orders_2025-11-01.json,182
bsh_orders_2025-11-01.csv,157
lms_orders_2025-11-01.csv,110


## E. Silver Table Data Quality, Optimization, and Transformation

In the silver layer, we refine the **raw bronze table** into a clean and consistent **single source of truth (silver table)**. 

This is where we standardize fields, enforce data quality, and prepare the dataset for downstream analytics.

In this section, you will:

- Apply data quality constraints  
- Clean and standardize fields  
- Enable liquid clustering for on the streaming table for optimized performance  

### E1. Create a New SQL File in your Pipeline

1. Click the kebab menu next to your `ingest_multiple_flows` folder and select **Create file**.
2. Select the language as **SQL**.
3. Name the file `silver_transformation.sql`.


### E2. Create Silver Transactions Table with Liquid Clustering and Data Quality Rules

1. Add the following code to your `silver_transformation.sql` file to create a Silver table with:
- a defined schema, 
- enforce consistent column types through `TRY_CAST` 
- and enable liquid clustering for performance

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
CREATE OR REFRESH STREAMING TABLE multi_flow_2_silver.orders_silver_flows_demo
(
  -- A: Define a fixed schema to prevent schema evolution.
  subsidiary_id   STRING,
  order_id        STRING,
  order_timestamp TIMESTAMP,
  order_date      DATE,
  customer_id     STRING,
  region          STRING,
  country         STRING,
  city            STRING,
  channel         STRING,
  sku             STRING,
  category        STRING,
  qty             INT,
  unit_price      DOUBLE,
  discount_pct    DOUBLE,
  total_amount    DOUBLE,
  coupon_code     STRING,

  -- B: Data quality constraints to drop or flag or drop invalid rows.
  CONSTRAINT qty_valid          EXPECT (qty >= 0) ON VIOLATION DROP ROW,
  CONSTRAINT total_amount_valid EXPECT (total_amount >= 0) ON VIOLATION DROP ROW,
  CONSTRAINT timestamp_not_null EXPECT (order_timestamp IS NOT NULL) ON VIOLATION FAIL UPDATE
)
-- C: Adds a table comment
COMMENT 'Clean and standardize data from the multiple-flow bronze table'

-- D: Enable liquid clustering to improve performance on common filters.
CLUSTER BY AUTO

AS
-- E: Select and clean data from the Bronze table. Uses TRY_CAST to enforce consistent types across all subsidiaries.
SELECT
  subsidiary_id,
  order_id,
  TRY_CAST(order_timestamp AS TIMESTAMP) AS order_timestamp, 
  TRY_CAST(order_date      AS DATE)      AS order_date,
  customer_id,
  region,
  country,
  city,
  channel,
  sku,
  category,
  TRY_CAST(qty          AS INT)    AS qty,
  TRY_CAST(unit_price   AS DOUBLE) AS unit_price,
  TRY_CAST(discount_pct AS DOUBLE) AS discount_pct,
  TRY_CAST(total_amount AS DOUBLE) AS total_amount,
  coupon_code
-- F: Incrementally reads data from the bronze table that contains data from three volumes
FROM STREAM multi_flow_1_bronze.orders_bronze_flows_demo;
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the code**
- A: `CREATE OR REFRESH STREAMING TABLE` - Creates or refreshes the **multi_flow_2_silver.orders_silver_flows_demo** Silver table and applies a fixed schema to prevent schema drift across multiple flows.

- B: `CONSTRAINT ... EXPECT`  Applies data quality rules that drop rows with invalid quantities, negative totals, or missing timestamps.

- C: `COMMENT`  Adds descriptive metadata explaining the purpose of the Silver table.

- D: `CLUSTER BY AUTO`  Enables automatic liquid clustering where Databricks intelligently chooses clustering keys to optimize your query performance. You can also specify your own clustering keys if you'd like.

- E: `TRY_CAST`  Enforces consistent column types across all subsidiaries by converting raw values into standardized data types.

- F: `SELECT ... FROM STREAM`  Incrementally reads, selects, and cleans records from the **multi_flow_1_bronze.orders_bronze_flows_demo** Bronze streaming table, which contains data from three separate volumes.

### E3. Run and Explore the Pipeline 
1. Select **Run pipeline** to create the Silver table.  
   - This run executes the transformation logic and applies the data quality expectations.

2. Explore the pipeline in the Lakeflow Pipeline Editor. Notice the following:
   - Since the pipeline has already ingested the source files into Bronze, **0** rows are processed in the Bronze table.
   - All **449** rows are processed in the Silver table.
   - In the **Expectations** column, select `3 met` to view the data quality rules. All rows pass the expectations.



#### Checkpoint

![Silver SDP Checkpoint](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/checkpoint_silver.png)

### E4. Explore the Silver Streaming table

1. Run the command to view the table metadata. When the results appear, notice the following:

- The silver table has the exact **column data types** that were defined.
- **Liquid clustering** is enabled:
  - In the **# Clustering Information** section no columns are specified since Databricks hasn't optimized the keys yet (requires historical query analysis on the table to optimize the clustered columns)
  - In the **Table Properties** row you will see cluster by auto is enabled (`clusterByAuto=true`).

**NOTE:** SDP supports Liquid clustering. Liquid clustering automatically organizes data based on frequently filtered columns to improve query performance. For more information, view the [Use liquid clustering for tables](https://docs.databricks.com/aws/en/delta/clustering).


In [0]:
%sql
DESCRIBE TABLE EXTENDED multi_flow_2_silver.orders_silver_flows_demo;

col_name,data_type,comment
subsidiary_id,string,null
order_id,string,null
order_timestamp,timestamp,null
order_date,date,null
customer_id,string,null
region,string,null
country,string,null
city,string,null
channel,string,null
sku,string,null


2. Run the code below to view the data in your **silver table**. 

    Confirm that the records look clean, standardized, and ready for gold-level analysis. 
    
    Look for consistent data types, valid numeric values, and properly cast timestamps and dates.


In [0]:
%sql
SELECT *
FROM multi_flow_2_silver.orders_silver_flows_demo;

subsidiary_id,order_id,order_timestamp,order_date,customer_id,region,country,city,channel,sku,category,qty,unit_price,discount_pct,total_amount,coupon_code
LMS,LMS_ORD_20251101_0001,2025-11-01T10:31:18.000Z,2025-11-01,CUST_04487,NA,US,Austin,online,SKU-4001,electronics,3,14.99,15.0,38.22,null
LMS,LMS_ORD_20251101_0002,2025-11-01T05:53:03.000Z,2025-11-01,CUST_02804,EU,UK,London,marketplace,SKU-5001,outdoor,4,9.99,15.0,33.97,null
LMS,LMS_ORD_20251101_0003,2025-11-01T21:41:50.000Z,2025-11-01,CUST_04968,EU,DE,Berlin,marketplace,SKU-1001,apparel,1,99.99,10.0,89.99,null
LMS,LMS_ORD_20251101_0004,2025-11-01T09:22:26.000Z,2025-11-01,CUST_00537,EU,DE,Berlin,mobile_app,SKU-3001,fitness,4,49.99,15.0,169.97,null
LMS,LMS_ORD_20251101_0005,2025-11-01T09:22:23.000Z,2025-11-01,CUST_03596,EU,DE,Berlin,marketplace,SKU-5001,outdoor,1,29.99,0.0,29.99,null
LMS,LMS_ORD_20251101_0006,2025-11-01T21:04:54.000Z,2025-11-01,CUST_03722,EU,DE,Berlin,mobile_app,SKU-1001,apparel,2,49.99,5.0,94.98,null
LMS,LMS_ORD_20251101_0007,2025-11-01T19:45:47.000Z,2025-11-01,CUST_00307,NA,US,Austin,marketplace,SKU-4001,electronics,2,14.99,15.0,25.48,null
LMS,LMS_ORD_20251101_0008,2025-11-01T19:14:02.000Z,2025-11-01,CUST_04361,NA,US,Denver,online,SKU-3001,fitness,3,14.99,5.0,42.72,null
LMS,LMS_ORD_20251101_0009,2025-11-01T05:23:55.000Z,2025-11-01,CUST_02755,EU,UK,London,mobile_app,SKU-2001,home,2,14.99,15.0,25.48,null
LMS,LMS_ORD_20251101_0010,2025-11-01T15:02:46.000Z,2025-11-01,CUST_03668,EU,UK,London,mobile_app,SKU-3001,fitness,2,49.99,5.0,94.98,null


## F. Business Intelligence Materialized Views

Materialized views in Lakeflow provide precomputed results that power fast, reliable analytics for downstream users. Unlike regular views, they automatically refresh as new data arrives, so stakeholders always see up-to-date insights without requiring on-demand computation.

**NOTE:** This section assumes prior familiarity with materialized views.


### F1. Create a New SQL File in your Pipeline

1. Click the kebab menu next to your `ingest_multiple_flows` folder and select **Create file**.
2. Select the language as **SQL**.
3. Name the file `gold_mvs.sql`.

### F2. Create Simple Gold Materialized Views

1. Next you will add two small simple materialized views for your consumers. These are only used to confirm that your silver data is clean and ready for analysis. Materialized views are not the focus here, so we will keep them simple.

   a. The first materialized view gives a **daily summary by subsidiary**, letting you quickly check revenue, units, and order counts over time.  

   b. The second view highlights **basic product performance** so you can see which categories and SKUs are selling within each subsidiary.


2. Copy the code below into your `gold_mvs.sql` file to create both materialized views.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
------------------------------------------
-- a. GOLD MATERIALIZED VIEW: DAILY SUBSIDIARY SCORECARD
-- Simple daily summary by subsidiary
------------------------------------------
CREATE OR REPLACE MATERIALIZED VIEW multi_flow_3_gold.mv_daily_subsidiary_scorecard_demo
AS
SELECT
  order_date,
  subsidiary_id,
  COUNT(DISTINCT order_id)    AS order_count,   -- how many unique orders occurred
  ROUND(SUM(total_amount),2)  AS total_revenue, -- total revenue for the day
  SUM(qty)                    AS total_units    -- total units sold
FROM multi_flow_2_silver.orders_silver_flows_demo
WHERE order_date IS NOT NULL
GROUP BY order_date, subsidiary_id;


------------------------------------------
-- b. GOLD MATERIALIZED VIEW: PRODUCT PERFORMANCE BY SUBSIDIARY
-- Basic units and revenue by product and subsidiary
------------------------------------------
CREATE OR REPLACE MATERIALIZED VIEW multi_flow_3_gold.mv_product_performance_by_subsidiary_demo
AS
SELECT
  subsidiary_id,
  category,
  sku,
  SUM(qty)                   AS units_sold,  -- total units sold for each SKU
  ROUND(SUM(total_amount),2) AS revenue      -- total revenue for each SKU
FROM multi_flow_2_silver.orders_silver_flows_demo
GROUP BY subsidiary_id, category, sku;
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>


### F3. Run and Explore the Pipeline

1. Select **Run pipeline** to create the two Gold materialized views.

2. Explore the pipeline in the Lakeflow Pipeline Editor. Notice the following:

   - Since the pipeline has already ingested the raw data into the **Bronze** and **Silver** streaming tables, and no new files were added, there are no new records for the streaming tables to process.

   - Both materialized views are created with **3** and **15** rows respectively.

   - In the **Tables** window at the bottom:  
     - Select the **Show and hide columns** button ![Show and Hide Columns](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/show_hide_columns_icon.png)  
     - Then show the **Incrementalization** column to see whether the materialized views were **fully recomputed** or **incrementally computed**.

   - On this initial run, both materialized views show **Full recompute** on the initial creation. Later, we will see these views compute incrementally.



#### Checkpoint

![MVs SDP Checkpoint](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/checkpoint_mvs_pipeline.png)

### F4. Display the Materialized Views

1. Querying the **multi_flow_3_gold.mv_daily_subsidiary_scorecard_demo** materialized view returns a daily scorecard for each subsidiary. 

    It summarizes order volume, total revenue, and total units so you can quickly compare performance across the three subsidiaries and order date.


In [0]:
%sql
SELECT *
FROM multi_flow_3_gold.mv_daily_subsidiary_scorecard_demo

order_date,subsidiary_id,order_count,total_revenue,total_units
2025-11-01,NSO,182,18620.31,535
2025-11-01,LMS,110,10550.83,306
2025-11-01,BSH,157,17994.31,467


2. The **multi_flow_3_gold.mv_product_performance_by_subsidiary_demo**  materialized view provides a simple product performance breakdown, showing which categories and SKUs are selling within each subsidiary. 

    It helps you compare units sold and revenue across product lines.


In [0]:
%sql
SELECT *
FROM multi_flow_3_gold.mv_product_performance_by_subsidiary_demo
ORDER BY subsidiary_id, category;

subsidiary_id,category,sku,units_sold,revenue
BSH,apparel,SKU-1001,83,2720.71
BSH,electronics,SKU-4001,80,2821.24
BSH,fitness,SKU-3001,88,3749.66
BSH,home,SKU-2001,91,3863.37
BSH,outdoor,SKU-5001,125,4839.33
LMS,apparel,SKU-1001,41,1567.09
LMS,electronics,SKU-4001,58,2058.92
LMS,fitness,SKU-3001,77,2394.55
LMS,home,SKU-2001,57,1725.22
LMS,outdoor,SKU-5001,73,2805.05


## G. Run the Spark Declarative Pipeline with New Files

Now that the pipeline is built, let's add the daily drop for the **2025-11-02** orders for each subsidiary.

### G1. Land a New File in Each Volume

1. Run the cell below to add the next daily file (**2025-11-02**) to each subsidiary volume.  

2. After the cell runs, confirm that each volume now contains two files: **2025-11-01** and **2025-11-02**.

In [0]:
## Copy a second CSV file into the bright_home_orders volume
marketplace_share_path = f'/Volumes/{your_marketplace_share_catalog_name}/v02/subsidiary_daily_orders'

copy_files(
    copy_from = f'{marketplace_share_path}/bright_home_orders', 
    copy_to = f'/Volumes/{my_catalog}/multi_flow_1_bronze/bright_home_orders', 
    n = 2
)

## Copy a second CSV file into the lumina_sports_orders volume
copy_files(
    copy_from = f'{marketplace_share_path}/lumina_sports_orders', 
    copy_to = f'/Volumes/{my_catalog}/multi_flow_1_bronze/lumina_sports_orders', 
    n = 2
)

## Copy a second JSON file into the northstar_outfitters_orders volume
copy_files(
    copy_from = f'{marketplace_share_path}/northstar_outfitters_orders', 
    copy_to = f'/Volumes/{my_catalog}/multi_flow_1_bronze/northstar_outfitters_orders', 
    n = 2
)


## List files in your volumes (confirm two files exist)
spark.sql(f'LIST "{my_vol_path}/bright_home_orders"').display()
spark.sql(f'LIST "{my_vol_path}/lumina_sports_orders"').display()
spark.sql(f'LIST "{my_vol_path}/northstar_outfitters_orders"').display()


----------------Loading files to user's volume: '/Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders'----------------
File number 1 - bsh_orders_2025-11-01.csv is already in the source volume "/Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders". Skipping file.
File number 2 - Copying file /Volumes/dbacademy_retail/v02/subsidiary_daily_orders/bright_home_orders/bsh_orders_2025-11-02.csv --> /Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders/bsh_orders_2025-11-02.csv.

----------------Loading files to user's volume: '/Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders'----------------
File number 1 - lms_orders_2025-11-01.csv is already in the source volume "/Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders". Skipping file.
File number 2 - Copying file /Volumes/dbacademy_retail/v02/subsidiary_daily_orders/lumina_sports_orders/lms_orders_2025-11-02.csv --> /Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders/lms_orders_2025-11-02.csv.

-

path,name,size,modification_time
/Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders/bsh_orders_2025-11-01.csv,bsh_orders_2025-11-01.csv,20219,1791030624000
/Volumes/dbx_gen_ai/multi_flow_1_bronze/bright_home_orders/bsh_orders_2025-11-02.csv,bsh_orders_2025-11-02.csv,24570,1791031059000


path,name,size,modification_time
/Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders/lms_orders_2025-11-01.csv,lms_orders_2025-11-01.csv,14248,1791030628000
/Volumes/dbx_gen_ai/multi_flow_1_bronze/lumina_sports_orders/lms_orders_2025-11-02.csv,lms_orders_2025-11-02.csv,21969,1791031064000


path,name,size,modification_time
/Volumes/dbx_gen_ai/multi_flow_1_bronze/northstar_outfitters_orders/nso_orders_2025-11-01.json,nso_orders_2025-11-01.json,68071,1791030633000
/Volumes/dbx_gen_ai/multi_flow_1_bronze/northstar_outfitters_orders/nso_orders_2025-11-02.json,nso_orders_2025-11-02.json,52748,1791031070000


### G2. Explore the New Daily Drop (2025-11-02) File in Each Volume
1. Simply run the cell below to count the number of records in the raw daily drop file **2025-11-02** within each volume.

2. Confirm that the output shows **502** total rows across all three subsidiaries for the **2025-11-02** orders drop.

| Volume                      | TotalRows | FileName                    |
|-----------------------------|-----------|------------------------------|
| bright_home_orders          | 191       | bsh_orders_2025-11-02.csv    |
| lumina_sports_orders        | 170       | lms_orders_2025-11-02.csv    |
| northstar_outfitters_orders | 141       | nso_orders_2025-11-02.json   |
| TOTAL                       | 502       |                              |

In [0]:
from pyspark.sql.functions import lit, sum as _sum

df_all = spark.sql(f"""
    SELECT 
        'bright_home_orders' AS Volume,
        COUNT(*) AS TotalRows,
        'bsh_orders_2025-11-02.csv' AS FileName
    FROM read_files('{my_vol_path}/bright_home_orders/bsh_orders_2025-11-02.csv')

    UNION ALL
    SELECT 
        'lumina_sports_orders' AS Volume,
        COUNT(*) AS TotalRows,
        'lms_orders_2025-11-02.csv' AS FileName
    FROM read_files('{my_vol_path}/lumina_sports_orders/lms_orders_2025-11-02.csv')

    UNION ALL
    SELECT 
        'northstar_outfitters_orders' AS Volume,
        COUNT(*) AS TotalRows,
        'nso_orders_2025-11-02.json' AS FileName
    FROM read_files('{my_vol_path}/northstar_outfitters_orders/nso_orders_2025-11-02.json')
""")

# Build TOTAL row
total_row = (
    df_all
    .agg(_sum("TotalRows").alias("TotalRows"))
    .withColumn("Volume", lit("TOTAL"))
    .withColumn("FileName", lit(""))
    .select("Volume", "TotalRows", "FileName")  # match column order
)

# Append TOTAL row to the bottom
df_with_total = df_all.unionByName(total_row)

display(df_with_total)


Volume,TotalRows,FileName
bright_home_orders,191,bsh_orders_2025-11-02.csv
lumina_sports_orders,170,lms_orders_2025-11-02.csv
northstar_outfitters_orders,141,nso_orders_2025-11-02.json
TOTAL,502,


### G3. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline to **incrementally** ingest, process and aggregate the **new daily sales drop**. Confirm it runs successfully. 

2. Explore the run in the Lakeflow Pipelines Editor.
  - Confirm **502** rows were ingested into the **bronze table** from the new daily orders drop in the three volumes.
  - Confirm all **502** were processed and passed the data quality checks in the **silver table**.
  - Confirm the materialized views:
    - Contain **6** and **15** rows respectively
    - Were both incrementally refreshed (**Incremental**). 
      - **NOTE:** For more information view the [Incremental refresh for materialized views](https://docs.databricks.com/aws/en/optimizations/incremental-refresh) documentation.

> **TROUBLESHOOTING:** If your pipeline does does not match the output below make sure you have landed the second file in each volume from the 'Land a New File in Each Volume' section above.

#### Checkpoint

![SDP Run 2 Files in All Volumes](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/checkpoint_run_daily_drop_2.png)


### G4. Explore the Final Pipeline Objects

1. Run the cell below to view the **bronze** table. 

    Notice that the table contains **951 rows** (The total number of rows after both runs)

In [0]:
%sql
SELECT *
FROM multi_flow_1_bronze.orders_bronze_flows_demo;

subsidiary_id,order_id,order_timestamp,customer_id,region,country,city,channel,sku,category,qty,unit_price,discount_pct,coupon_code,total_amount,order_date,source_file,file_mod_time
BSH,BSH_ORD_20251102_0001,2025-11-02 17:44:57,CUST_04394,NA,US,Denver,online,SKU-3001,fitness,2,14.99,10,null,26.98,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0002,2025-11-02 15:55:09,CUST_01787,NA,US,Seattle,marketplace,SKU-3001,fitness,1,9.99,10,null,8.99,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0003,2025-11-02 14:57:02,CUST_04718,NA,US,Seattle,marketplace,SKU-1001,apparel,4,49.99,10,null,179.96,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0004,2025-11-02 21:55:28,CUST_03357,NA,US,Seattle,online,SKU-3001,fitness,2,49.99,10,null,89.98,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0005,2025-11-02 05:42:41,CUST_00206,EU,DE,Berlin,marketplace,SKU-4001,electronics,2,99.99,15,null,169.98,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0006,2025-11-02 19:21:27,CUST_00886,NA,US,Denver,online,SKU-3001,fitness,2,9.99,10,null,17.98,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0007,2025-11-02 00:26:53,CUST_00285,EU,UK,London,marketplace,SKU-5001,outdoor,3,14.99,10,null,40.47,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0008,2025-11-02 15:57:45,CUST_04771,EU,DE,Berlin,online,SKU-4001,electronics,5,14.99,5,null,71.2,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0009,2025-11-02 07:34:59,CUST_04970,NA,US,Denver,online,SKU-5001,outdoor,1,99.99,5,null,94.99,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z
BSH,BSH_ORD_20251102_0010,2025-11-02 01:39:45,CUST_01464,EU,DE,Berlin,marketplace,SKU-2001,home,4,9.99,15,null,33.97,2025-11-02,bsh_orders_2025-11-02.csv,2026-10-03T12:37:39.000Z


2. Count the number of rows ingested by each **source_file** in the **bronze** streaming table. 

    Notice that we can easily examine how many rows were ingested by each source file (**daily orders drop**).

In [0]:
%sql
SELECT source_file, count(*) AS TotalRows
FROM multi_flow_1_bronze.orders_bronze_flows_demo
GROUP BY source_file
ORDER BY source_file;

source_file,TotalRows
bsh_orders_2025-11-01.csv,157
bsh_orders_2025-11-02.csv,191
lms_orders_2025-11-01.csv,110
lms_orders_2025-11-02.csv,170
nso_orders_2025-11-01.json,182
nso_orders_2025-11-02.json,141


3. View the data in the **silver table**. Notice that the data is clean, adheres to our defined schema and contains our 'single source of truth'.

In [0]:
%sql
SELECT *
FROM multi_flow_2_silver.orders_silver_flows_demo;

subsidiary_id,order_id,order_timestamp,order_date,customer_id,region,country,city,channel,sku,category,qty,unit_price,discount_pct,total_amount,coupon_code
NSO,NSO_ORD_20251102_0001,2025-11-02T13:44:06.000Z,2025-11-02,CUST_04421,EU,UK,London,online,SKU-3001,fitness,2,49.99,5.0,94.98,null
NSO,NSO_ORD_20251102_0002,2025-11-02T21:20:14.000Z,2025-11-02,CUST_03199,NA,US,Denver,marketplace,SKU-4001,electronics,5,29.99,10.0,134.95,null
NSO,NSO_ORD_20251102_0003,2025-11-02T13:47:18.000Z,2025-11-02,CUST_00581,NA,US,Denver,online,SKU-1001,apparel,3,29.99,10.0,80.97,null
NSO,NSO_ORD_20251102_0004,2025-11-02T05:53:56.000Z,2025-11-02,CUST_02327,NA,US,Seattle,mobile_app,SKU-3001,fitness,1,49.99,0.0,49.99,null
NSO,NSO_ORD_20251102_0005,2025-11-02T10:29:58.000Z,2025-11-02,CUST_02264,NA,US,Denver,marketplace,SKU-1001,apparel,1,14.99,15.0,12.74,null
NSO,NSO_ORD_20251102_0006,2025-11-02T09:26:33.000Z,2025-11-02,CUST_00638,NA,US,Denver,mobile_app,SKU-2001,home,4,9.99,15.0,33.97,null
NSO,NSO_ORD_20251102_0007,2025-11-02T12:32:09.000Z,2025-11-02,CUST_03981,EU,DE,Berlin,online,SKU-5001,outdoor,3,49.99,5.0,142.47,null
NSO,NSO_ORD_20251102_0008,2025-11-02T19:40:49.000Z,2025-11-02,CUST_04323,NA,US,Denver,online,SKU-2001,home,3,9.99,10.0,26.97,null
NSO,NSO_ORD_20251102_0009,2025-11-02T15:31:13.000Z,2025-11-02,CUST_04065,NA,US,Denver,marketplace,SKU-3001,fitness,2,29.99,10.0,53.98,null
NSO,NSO_ORD_20251102_0010,2025-11-02T20:56:46.000Z,2025-11-02,CUST_00640,NA,US,Seattle,mobile_app,SKU-3001,fitness,5,9.99,15.0,42.46,null


4. View the gold level materialized view **mv_daily_subsidiary_scorecard_demo**. 

    Notice downstream consumers can easily examine orders by date for each subsidiary.

In [0]:
%sql
SELECT *
FROM multi_flow_3_gold.mv_daily_subsidiary_scorecard_demo
ORDER BY order_date, subsidiary_id;

order_date,subsidiary_id,order_count,total_revenue,total_units
2025-11-01,BSH,157,17994.31,467
2025-11-01,LMS,110,10550.83,306
2025-11-01,NSO,182,18620.31,535
2025-11-02,BSH,191,24052.46,576
2025-11-02,LMS,170,19646.81,514
2025-11-02,NSO,141,13864.85,429


5. View the gold level materialized view **mv_product_performance_by_subsidiary_demo**. 

    Notice downstream consumers can easily examine detailed order metrics by each **subsidiary_id** and **sku**. 

In [0]:
%sql
SELECT *
FROM multi_flow_3_gold.mv_product_performance_by_subsidiary_demo
ORDER BY subsidiary_id, category;

subsidiary_id,category,sku,units_sold,revenue
BSH,apparel,SKU-1001,204,7749.79
BSH,electronics,SKU-4001,177,6047.8
BSH,fitness,SKU-3001,203,8635.56
BSH,home,SKU-2001,240,11147.45
BSH,outdoor,SKU-5001,219,8466.17
LMS,apparel,SKU-1001,156,5495.95
LMS,electronics,SKU-4001,166,6353.42
LMS,fitness,SKU-3001,191,7260.94
LMS,home,SKU-2001,156,5109.04
LMS,outdoor,SKU-5001,151,5978.29


## H. Introduction to Adding Tags to Bronze, Silver and Gold Objects
#### This requires the necessary permissions to add tags

In this step you add semantic metadata to your tables and materialized views. Tags make it easier to organize, search and govern objects in Unity Catalog. They help downstream teams quickly understand what each object represents and how it should be used.

You apply two types of tags:

- **Custom demo tags**  
  These describe the department that owns the data and the quality level in the medallion architecture.  
  Examples:  
  - `demo_tag_Department = 'Sales'`  
  - `demo_tag_Quality = 'bronze' | 'silver' | 'gold'`

- **System tags**  
  These are built-in Unity Catalog tags. Here you add the `system.Certified` tag to identify trusted, production-ready objects.


**NOTE**: For more information view the [Apply tags to Unity Catalog securable objects](https://docs.databricks.com/aws/en/database-objects/tags)

1. The code below creates the following tags:
  - Updates the **bronze table** with department and quality tags so users can see this is raw, ingested data owned by Sales.
  - Updates the **silver table** with the same tags, then adds the `system.Certified` tag to mark it as a clean, trusted dataset.
  - Updates both **gold materialized views** with department and quality tags, then marks each one as `system.Certified` because these are curated analytics objects intended for broad consumption.

**NOTE:** [ALTER TABLE]()

In [0]:
%sql

------------------------------------
-- Bronze table tags
----------------------------------
ALTER TABLE multi_flow_1_bronze.orders_bronze_flows_demo
SET TAGS (
  'demo_tag_Department' = 'Sales',
  'demo_tag_Quality' = 'bronze'
);


----------------------------------
-- Silver table tags
----------------------------------
ALTER TABLE multi_flow_2_silver.orders_silver_flows_demo
SET TAGS (
  'demo_tag_Department' = 'Sales',
  'demo_tag_Quality' = 'silver'
);


----------------------------------
-- Materialized views table tags
------------------------------------
ALTER TABLE multi_flow_3_gold.mv_product_performance_by_subsidiary_demo
SET TAGS (
  'demo_tag_Department' = 'Sales',
  'demo_tag_Quality' = 'gold'
);

ALTER TABLE multi_flow_3_gold.mv_daily_subsidiary_scorecard_demo
SET TAGS (
  'demo_tag_Department' = 'Sales',
  'demo_tag_Quality' = 'gold'
);

2. After you add tags to a table or materialized view, you can query them directly using the **information_schema.table_tags** view within the specific catalog, or the **system.information_schema** schema. 

    The query below queries **your-catalog.information_schema**.

In [0]:
%sql
SELECT *
FROM information_schema.table_tags;

catalog_name,schema_name,table_name,tag_name,tag_value
dbx_gen_ai,default,llama4_payload,product,ai-gateway
dbx_gen_ai,multi_flow_3_gold,mv_product_performance_by_subsidiary_demo,demo_tag_Department,Sales
dbx_gen_ai,multi_flow_3_gold,mv_product_performance_by_subsidiary_demo,demo_tag_Quality,gold
dbx_gen_ai,multi_flow_3_gold,mv_daily_subsidiary_scorecard_demo,demo_tag_Department,Sales
dbx_gen_ai,multi_flow_3_gold,mv_daily_subsidiary_scorecard_demo,demo_tag_Quality,gold
dbx_gen_ai,multi_flow_1_bronze,orders_bronze_flows_demo,demo_tag_Department,Sales
dbx_gen_ai,multi_flow_1_bronze,orders_bronze_flows_demo,demo_tag_Quality,bronze
dbx_gen_ai,multi_flow_2_silver,orders_silver_flows_demo,demo_tag_Quality,silver
dbx_gen_ai,multi_flow_2_silver,orders_silver_flows_demo,demo_tag_Department,Sales


3. You can also view tags in **Catalog Explorer**. Follow these steps to inspect the tags on one of your objects:

   a. Right click **Catalog** and select **Open in New Tab**  

   b. Navigate to your catalog  

   c. Open the **multi_flow_3_gold** schema  

   d. Select the **mv_daily_subsidiary_scorecard_demo** materialized view  

   e. In the right pane, locate the **Tags** section and confirm the three tags that were added

#### Checkpoint
![Tagging Checkpoint](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.4/images/20260921T154804Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/multi_flow/checkpoint_tagging.png)

## I. (OPTIONAL) Land Additional Files in your Volumes

If you'd like to continue practicing, use the demonstration function `copy_files` to dynamically add another JSON file (file number 3) to your cloud storage location.  

**NOTES:** 
- Ensure the variables you defined at the start of this lab: `your_marketplace_share_catalog_name`, `my_catalog` are still active for the function to work properly.
- There are a total of 7 available files you can continue practicing with.

In [0]:
%skip

## Copy a second CSV file into the bright_home_orders volume
marketplace_share_path = f'/Volumes/{your_marketplace_share_catalog_name}/v02/subsidiary_daily_orders'

copy_files(
    copy_from = f'{marketplace_share_path}/bright_home_orders', 
    copy_to = f'/Volumes/{my_catalog}/multi_flow_1_bronze/bright_home_orders', 
    n = 3 # <-- Add a third file to the volume
)

## Copy a second CSV file into the lumina_sports_orders volume
copy_files(
    copy_from = f'{marketplace_share_path}/lumina_sports_orders', 
    copy_to = f'/Volumes/{my_catalog}/multi_flow_1_bronze/lumina_sports_orders', 
    n = 3 # <-- Add a third file to the volume
)

## Copy a second JSON file into the northstar_outfitters_orders volume
copy_files(
    copy_from = f'{marketplace_share_path}/northstar_outfitters_orders', 
    copy_to = f'/Volumes/{my_catalog}/multi_flow_1_bronze/northstar_outfitters_orders', 
    n = 3 # <-- Add a third file to the volume
)


## List files in your volumes
spark.sql(f'LIST "{my_vol_path}/bright_home_orders"').display()
spark.sql(f'LIST "{my_vol_path}/lumina_sports_orders"').display()
spark.sql(f'LIST "{my_vol_path}/northstar_outfitters_orders"').display()

## J. Clean up
1. Feel free to delete the schemas you create in this demonstration by running cell below and confirming the delete (**Y**).

In [0]:
delete_schemas(
    catalog = my_catalog, ## <--- Your catalog name using the variable you set earlier
    schemas = ['multi_flow_1_bronze', 'multi_flow_2_silver', 'multi_flow_3_gold']
)

Are you sure you want to delete these schemas in catalog 'dbx_gen_ai': ['multi_flow_1_bronze', 'multi_flow_2_silver', 'multi_flow_3_gold']? (Y/N):  Y

Deleting schemas ['multi_flow_1_bronze', 'multi_flow_2_silver', 'multi_flow_3_gold'] in catalog 'dbx_gen_ai'...
Schemas deleted (if they existed).


2. Delete your Spark Declarative Pipeline through the **Jobs & Pipelines** UI.

## K. Summary and Key Takeaways 

- **Multi-Flow Ingestion**: Successfully ingested data from three different subsidiaries (`CSV` and `JSON` formats) into a single bronze streaming table using separate flows
- **Schema Standardization**: Resolved format conflicts by casting all columns to STRING in bronze, then applying proper data types in silver
- **Data Quality & Performance**: Implemented constraint-based data quality checks and enabled liquid clustering for optimized query performance
- **Incremental Processing**: Demonstrated true incremental processing across the entire medallion architecture with automatic materialized view refresh

#### Business Value Delivered

The pipeline processed **951 total records** from six source files across two daily drops, creating a unified view of sales data that enables:
- Cross-subsidiary performance analysis
- Real-time business intelligence through auto-refreshing materialized views
- Scalable data quality enforcement as new subsidiaries are added

This architecture provides a foundation for enterprise-scale data consolidation while maintaining data lineage, quality, and performance optimization.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>